In [ ]:
# #1: Cài đặt
# !pip install transformers evaluate jiwer wandb -q

In [ ]:
#2: Imports
import os, gc, json, time, glob
import numpy as np
import torch
import wandb
import psutil
from torch.utils.data import Dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from evaluate import load as load_metric

In [ ]:
#3: Environment
def print_memory(label=''):
    ram = psutil.virtual_memory()
    if torch.cuda.is_available():
        vram       = torch.cuda.memory_allocated() / 1e9
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[{label}] RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB | VRAM: {vram:.1f}/{vram_total:.1f}GB')
    else:
        print(f'[{label}] RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CPU only')
print_memory('start')

In [ ]:
#4: Config
VIVOS_DIR  = '/kaggle/input/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr/vivos'
MODEL_DIR  = '/kaggle/input/datasets/thientmai220205/wav2vec2-vimd-checkpoint'

STEP_SIZE  = 10
EVAL_BATCH = 1

In [ ]:
#5: WandB
wandb.login(key='')
run = wandb.init(
    project='',
    entity='',
    name='',
    resume='allow',
)

In [ ]:
#6: Load Processor + Model
print('Loading processor...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
print(f'Vocab size: {len(processor.tokenizer)}')

print('Loading model...')
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
print(f'Model loaded on {device}')
print_memory('after model')

In [ ]:
#7: Load VIVOS
import soundfile as sf
import numpy as np
import torch

def load_vivos_split(vivos_dir, split):
    split_dir    = os.path.join(vivos_dir, split)
    prompts_path = os.path.join(split_dir, 'prompts.txt')
    waves_dir    = os.path.join(split_dir, 'waves')

    print(f'  split_dir:    {split_dir}')
    print(f'  prompts_path: {prompts_path}')
    print(f'  waves_dir:    {waves_dir}')
    print(f'  prompts exists: {os.path.exists(prompts_path)}')

    samples = []
    missing = 0
    with open(prompts_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts    = line.split(' ', 1)
            file_id  = parts[0]
            text     = parts[1].lower() if len(parts) > 1 else ''
            speaker  = file_id.split('_')[0]
            wav_path = os.path.join(waves_dir, speaker, f'{file_id}.wav')
            if os.path.exists(wav_path):
                samples.append({'path': wav_path, 'text': text})
            else:
                missing += 1

    print(f'  VIVOS {split}: {len(samples):,} | missing: {missing}')
    return samples

vivos_train = load_vivos_split(VIVOS_DIR, 'train')
vivos_test  = load_vivos_split(VIVOS_DIR, 'test')
vivos_all   = vivos_train + vivos_test

print(f'\nVIVOS total: {len(vivos_all):,}')
print(f'  train: {len(vivos_train):,}')
print(f'  test:  {len(vivos_test):,}')


class VIVOSWavDataset(torch.utils.data.Dataset):
    def __init__(self, samples, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        array, sr = sf.read(sample['path'])
        array = array.astype(np.float32)
        if len(array.shape) > 1:
            array = array.mean(axis=1)
        if sr != 16000:
            import librosa
            array = librosa.resample(array, orig_sr=sr, target_sr=16000)

        input_values = self.processor(
            array, sampling_rate=16000
        ).input_values[0]
        labels = self.processor.tokenizer(sample['text']).input_ids

        return {
            'input_values': np.array(input_values, dtype=np.float32),
            'labels':       labels,
        }

vivos_dataset = VIVOSWavDataset(vivos_all, processor)
print(f'Dataset ready: {len(vivos_dataset):,} samples')
print_memory('after dataset')

In [ ]:
#8: Evaluation VIVOS
wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

n          = len(vivos_dataset)
all_preds  = []
all_refs   = []
all_times  = []
step_preds = []
step_refs  = []
step_times = []

print(f'Starting evaluation on {n:,} samples (step={STEP_SIZE}, batch={EVAL_BATCH})...')
print('=' * 80)

for start in range(0, n, EVAL_BATCH):
    end   = min(start + EVAL_BATCH, n)
    batch = [vivos_dataset[i] for i in range(start, end)]

    input_list   = [{'input_values': b['input_values']} for b in batch]
    padded       = processor.pad(input_list, padding=True, return_tensors='pt')
    input_values = padded['input_values'].to(device)

    label_list     = [{'input_ids': b['labels']} for b in batch]
    padded_labels  = processor.tokenizer.pad(label_list, padding=True, return_tensors='pt')
    labels         = padded_labels['input_ids'].to(device)
    attention_mask = padded_labels['attention_mask'].to(device)
    labels         = labels.masked_fill(attention_mask.ne(1), -100)
    labels[labels >= len(processor.tokenizer)] = -100

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        output = model(input_values=input_values)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    per_sample_time = elapsed / len(batch)
    for _ in batch:
        step_times.append(per_sample_time)
        all_times.append(per_sample_time)

    pred_ids_batch = torch.argmax(output.logits, dim=-1)
    pred_strs      = processor.batch_decode(pred_ids_batch)

    for i, b in enumerate(batch):
        pred_str  = pred_strs[i]
        label_ids = [int(x) for x in b['labels'] if int(x) != -100]
        label_str = processor.tokenizer.decode(label_ids)
        step_preds.append(pred_str)
        step_refs.append(label_str)
        all_preds.append(pred_str)
        all_refs.append(label_str)

    if len(all_preds) % STEP_SIZE == 0 or end == n:
        step_num     = len(all_preds)
        step_wer     = wer_metric.compute(predictions=step_preds, references=step_refs)
        step_cer     = cer_metric.compute(predictions=step_preds, references=step_refs)
        step_avg_inf = np.mean(step_times)

        print(f'[Step {step_num:>5}] samples={len(step_preds):>5} | WER={step_wer*100:.2f}% | CER={step_cer*100:.2f}% | Avg Inf. Time={step_avg_inf:.4f}s/sample')
        print(f'Pred label: {step_preds[-1]}')
        print(f'True label: {step_refs[-1]}')
        print('.' * 80)

        wandb.log({
            'step/samples':   step_num,
            'step/wer':       step_wer,
            'step/cer':       step_cer,
            'step/infer_sec': step_avg_inf,
        })

        step_preds = []
        step_refs  = []
        step_times = []

In [ ]:
#9: Final Results
final_wer     = wer_metric.compute(predictions=all_preds, references=all_refs)
final_cer     = cer_metric.compute(predictions=all_preds, references=all_refs)
final_avg_inf = np.mean(all_times)
total_time    = sum(all_times)

print()
print('EVALUATION RESULTS')
print('=' * 40)
print(f'Dataset         : VIVOS dataset')
print(f'Samples         : {len(all_preds)}')
print(f'WER             : {final_wer*100:.2f}%')
print(f'CER             : {final_cer*100:.2f}%')
print(f'Avg Inf. Time   : {final_avg_inf:.4f} seconds/sample')
print('=' * 40)
print('Logged final summary results to WandB.')

wandb.log({
    'final/total_samples': len(all_preds),
    'final/wer':           final_wer,
    'final/cer':           final_cer,
    'final/avg_infer_sec': final_avg_inf,
    'final/total_time':    total_time,
})
wandb.finish()